# Reproduce numbers: `glocalkd`

Verifies 2 MASTER row(s) for this experiment by reloading frozen artifacts.


In [ ]:
from __future__ import annotations
import json, sys
from pathlib import Path
CWD = Path.cwd().resolve()
REPO = CWD if (CWD / 'abrg').is_dir() else CWD.parent
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / 'chapter_a' / 'scripts'))
from auc_from_artifact import extract_auc, close6, as_float
EXP = 'glocalkd'
CFG = json.loads((REPO / 'chapter_a' / 'reproduce' / 'by_experiment' / EXP / 'reproduce_config.json').read_text())
failed = []
for r in CFG['rows']:
    row = dict(r)
    row['experiment'] = EXP
    art, mode = extract_auc(row)
    master = as_float(row['auc_floor'])
    if row['detector'] == 'HGB_mean_raw_auc':
        master = as_float(row['raw_auc'])
    ok = art is not None and close6(master, art)
    print(f"{row['detector']:40s} master={master} art={art} mode={mode} ok={ok}")
    if not ok:
        failed.append({'detector': row['detector'], 'master': master, 'art': art, 'mode': mode})
assert not failed, failed
print('OK', EXP)
